# SIMT 核函数开发

本节把第 6 章的模型要素组装成标准开发骨架，并引入处理大数据量的 grid-stride 模式。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 三层开发骨架

```text
Host 侧                    外层 JIT Kernel               SIMT 函数
准备 Tensor、               section_vector 中            逐线程逻辑：
算 block 数、               以 f[threads](...)           索引映射 + 边界保护 +
kernel[None, blocks](...)  启动线程块                   原子/协作
```

- **Host**：准备静态 Shape 的 NPU Tensor，选择不超过当前 Stream 可用 Vector 核数的 blocks，用 grid-stride 覆盖数据，不能无限按 ceil(n/THREADS) 增加核数；
- **外层 Kernel**：`with pl.section_vector(): simt_func[THREADS](...)`，负责 Tile/搬运/流水（本章场景直访 Tensor，无需 Tile）；
- **SIMT 函数**：每线程执行一次，索引映射数据、边界判断保护、必要时原子协作。


## 2. grid-stride：线程数少于数据量

数据 12288 行、线程 48 核 × 256 = 12288 恰好一一对应是巧合；通用写法是**网格跨步循环**：

```python
first_row = pl.simt.block_idx().x * pl.simt.block_dim().x + pl.simt.thread_idx().x
row_stride = pl.simt.grid_dim().x * pl.simt.block_dim().x     # 总线程数
for row in pl.range(first_row, row_count, row_stride):        # 每线程跨步认领多行
    ...
```

要点：

1. `row_stride` 是**全局线程总数**（grid_x × 每 Block 的线程数；本例 Block 为一维），保证各行恰好被一个线程处理、不重不漏；
2. 循环边界用 `row_count`（运行时入参），天然支持尾部截断；
3. 与"加大核数凑线程"相比，grid-stride 让线程数固定、数据量自由——这是生产代码的默认形态。


## 3. SIMT 函数内能用的"工具箱"

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">类别</th><th style="text-align:left;">接口（前缀 pl.simt.）</th><th style="text-align:left;">用途</th></tr>
<tr><td>索引</td><td>block_idx / thread_idx / block_dim / grid_dim / linear_thread_idx</td><td>数据映射</td></tr>
<tr><td>标量数学</td><td>abs / min / max / sqrt / rsqrt / fma</td><td>逐线程计算</td></tr>
<tr><td>超越函数</td><td>exp / log / sin / cos / tanh …</td><td>激活等</td></tr>
<tr><td>取整/判断</td><td>rint / round / floor / ceil / trunc / isnan / isinf</td><td>数值处理</td></tr>
<tr><td>类型转换</td><td>cast / bitcast</td><td>数值转换/位重解释</td></tr>
<tr><td>原子操作</td><td>atomic_add / atomic_max …</td><td>同位置竞争</td></tr>
<tr><td>同步</td><td>syncthreads</td><td>块内屏障</td></tr>
</table>


### 证明覆盖：把整数除法写出来

假设 2 个 Block、每块 4 个线程，数据长度 19。全局线程 t=0～7，第 0 轮处理 0～7，第 1 轮处理 8～15，第 2 轮只有线程 0～2 处理 16～18。对任意 i，唯一拥有者为 t=i%8、轮次 k=i//8。

若误把 stride 写成核数 2，线程 0 会访问 0、2、4，与线程 2 的第一次访问冲突；若只有 if 而没有循环，8～18 无人处理。先用这样的整数算例验证映射，再扩大输入。这个证明不依赖线程实际调度顺序。


## 4. 课后练习

1. （填空）grid-stride 的步长 = `______.x * ______.x`（写表达式）。
2. （选择）20000 行数据、8 核 × 256 线程：grid-stride 下最忙的线程处理多少行？
   A. 8<br/>B. 10<br/>C. 256
3. （实践）把第 6 章实践一的逐线程加法改写为 grid-stride 版（数据量改为 [1, 10000]），说明循环边界如何处理尾部。

In [ ]:
!cat ./answer/07.02_answer.txt

## 5. 本节小结

- 三层骨架：**Host 算规模 → 外层 Kernel 启动 → SIMT 逐线程逻辑**；
- grid-stride：步长=全局线程数，行数自由、不重不漏，生产默认；
- SIMT 工具箱七类接口（索引/标量/超越/取整/转换/原子/同步）。